<a href="https://colab.research.google.com/github/nisithprusty-cyber/AI_PROJECT-AUTONOMOUS_JOB_APPLICATION-/blob/main/2301020310-day-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu pymupdf google-generativeai beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 26.5 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI Studio API Key: ")

Enter your Google AI Studio API Key: ··········


In [3]:
from google.colab import files
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)

Saving Akash Resume .pdf to Akash Resume .pdf
Uploaded: Akash Resume .pdf


In [4]:
import fitz  # PyMuPDF

def load_resume(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

resume_text = load_resume(pdf_path)
print("\nResume Loaded Successfully!")


Resume Loaded Successfully!


In [5]:
import re

def preprocess_resume(text):
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\+?\d[\d\s\-]{8,}', '', text)
    return text

resume_text = preprocess_resume(resume_text)

In [6]:
def create_chunks(text):
    sections = re.split(r'\n+', text)
    chunks = []

    for sec in sections:
        sec = sec.strip()
        if len(sec) > 40:
            chunks.append(sec)

    return chunks

chunks = create_chunks(resume_text)
print("Total Chunks:", len(chunks))

Total Chunks: 18


In [7]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_chunks(chunks):
    return embed_model.encode(chunks)

embeddings = embed_chunks(chunks)
print("Embeddings Created")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings Created


In [8]:
import faiss
import numpy as np

def build_vector_store(embeddings):
    dimension = len(embeddings[0])
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings))
    return index

index = build_vector_store(embeddings)
print("Vector DB Ready")

Vector DB Ready


In [9]:
def query_vector_store(index, chunks, query, k=5):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    return [chunks[i] for i in indices[0]]

In [10]:
import requests
from bs4 import BeautifulSoup

def scrape_job_description(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text

In [ ]:
def extract_skills(text):
    # This function uses the generative model to extract skills.
    # Ensure 'model' from google.generativeai is initialized before calling this function.
    prompt = f"Extract all technical skills from the following text, listing them as a comma-separated string. If no skills are found, return 'No skills identified'.\n\nText:\n{text}\n\nSkills (comma-separated):"
    response = model.generate_content(prompt)
    return response.text

print("\nExtracting Job Skills...")
job_skills = extract_skills(job_text)
print("Job Skills:", job_skills)

In [13]:
def compare_skills(resume_skills, job_skills):
    prompt = f"""
    Compare these:

    Resume Skills:
    {resume_skills}

    Job Skills:
    {job_skills}

    Return ONLY:

    Matching Skills: []
    Missing Skills: []
    Match Percentage: %
    """
    response = model.generate_content(prompt)
    return response.text

In [14]:
def final_decision(result):
    prompt = f"""
    Based on:

    {result}

    If match percentage >= 70% → SELECTED
    Else → NOT SELECTED

    Output ONLY:
    Decision: SELECTED or NOT SELECTED
    """
    response = model.generate_content(prompt)
    return response.text

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
model = genai.GenerativeModel("gemini-2.5-flash")

job_url = input("\nEnter Job URL: ")

print("\nScraping Job Description...")
job_text = scrape_job_description(job_url)

relevant_chunks = query_vector_store(index, chunks, "skills experience technologies", k=5)
filtered_resume_text = " ".join(relevant_chunks)

In [ ]:
print("\nExtracting Resume Skills...")
resume_skills = extract_skills(filtered_resume_text)
print("Resume Skills:", resume_skills)

In [ ]:
print("\nPlease paste the Job Description here (Press Enter twice to stop):")
job_lines = []
while True:
    line = input()
    if line == "":
        break
    job_lines.append(line)

job_text = "\n".join(job_lines)
print("\nJob Description captured.")

In [ ]:
print("\nExtracting Job Skills...")
job_skills = extract_skills(job_text)
print("Job Skills:", job_skills)

In [ ]:
print("\nComparing Skills...")
comparison = compare_skills(resume_skills, job_skills)
print("\n", comparison)

print("\nFinal Decision...")
decision = final_decision(comparison)
print(decision)